In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df=pd.read_csv("../data/train.csv")

In [ ]:
df.sample(4)

In [ ]:
def calc_distance(pickup_lat, pickup_long, dropoff_lat, dropoff_long):
    
    pickup_lat = np.radians(pickup_lat)
    pickup_long = np.radians(pickup_long)
    
    dropoff_lat = np.radians(dropoff_lat)
    dropoff_long = np.radians(dropoff_long)
    
    dlat = dropoff_lat - pickup_lat
    dlong = dropoff_long - pickup_long
    
    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(pickup_lat)
        * np.cos(dropoff_lat)
        * np.sin(dlong / 2) ** 2
    )
    
    c = 2 * np.arcsin(np.sqrt(a))
    
    distance_km = 6371 * c
    
    return distance_km

In [ ]:
df['distance_km'] = df.apply(
    lambda x: calc_distance(
        x['pickup_latitude'],
        x['pickup_longitude'],
        x['dropoff_latitude'],
        x['dropoff_longitude']
    ),
    axis=1
)

In [ ]:
df.sample(5)

In [ ]:
import seaborn as sns

In [ ]:
sns.heatmap(df.corr(numeric_only=True), annot=True)

In [ ]:
df['vendor_id'].value_counts()

In [ ]:
print(df.columns.tolist())

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# -----------------------------
# Load data
# -----------------------------



# -----------------------------
# Datetime features
# -----------------------------

df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

df['pickup_hour'] = df['pickup_datetime'].dt.hour
df['pickup_dayofweek'] = df['pickup_datetime'].dt.dayofweek
df['pickup_month'] = df['pickup_datetime'].dt.month

df['is_weekend'] = (df['pickup_dayofweek'] >= 5).astype(int)

df['is_rush_hour'] = (
    df['pickup_hour'].isin([7, 8, 9, 17, 18, 19])
).astype(int)


# -----------------------------
# Convert categorical column
# -----------------------------

df['store_and_fwd_flag'] = (
    df['store_and_fwd_flag'] == 'Y'
).astype(int)


# -----------------------------
# Features
# -----------------------------

features = [
    'vendor_id',
    'passenger_count',
    'pickup_longitude',
    'pickup_latitude',
    'dropoff_longitude',
    'dropoff_latitude',
    'distance_km',
    'pickup_hour',
    'pickup_dayofweek',
    'pickup_month',
    'is_weekend',
    'is_rush_hour',
    'store_and_fwd_flag'
]

X = df[features]
y = df['trip_duration']


# -----------------------------
# Train Test Split
# -----------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# -----------------------------
# Model
# -----------------------------

model = LinearRegression()

model.fit(X_train, y_train)


# -----------------------------
# Prediction
# -----------------------------

y_pred = model.predict(X_test)


# -----------------------------
# Evaluation
# -----------------------------

mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

r2 = r2_score(y_test, y_pred)


print("MAE :", mae)
print("RMSE:", rmse)
print("R2  :", r2)